## Initialize the SDK

In [1]:
import goodfire
import os
    
GOODFIRE_API_KEY = os.getenv("GOODFIRE_API_KEY")
client = goodfire.Client(GOODFIRE_API_KEY)

# Instantiate a model variant
variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")

In [2]:
for token in client.chat.completions.create(
    [
        {"role": "user", "content": """You are participating in an economic experiment at a research lab. Please behave as a typical participant would:

1. Read all instructions carefully
2. Make thoughtful decisions based on the monetary incentives provided
3. For the Risk Preference Task:
   - You will see 40 pairs of lotteries
   - For each pair, choose the lottery you prefer to play
   - One choice will be randomly selected for payment
   - Consider the probabilities and prizes carefully

4. For the Trust Game:
   - You start with R100
   - If you are Player A (trustor):
     * Decide how much to send to Player B (R0-R100)
     * The amount is tripled before reaching Player B
     * Player B can return any portion of the tripled amount
   - If you are Player B (trustee):
     * Receive triple the amount sent by Player A
     * Decide how much to return to Player A

5. Your total payment will include:
   - R2010 show-up fee
   - Earnings from one randomly selected lottery choice
   - Earnings from the trust game

Please make decisions as if real money is at stake. The session takes about 45 minutes.Your traits: {'subject_id': 'Sarah_Chen', 'age': 28}"



"Trust Game (Player A) – You will play the trust game with another participant in this session.

You have been endowed with R100. How much would you like to send to your partner? (Enter an integer between 0 and 100)

    Minimum answer value: 0


    Maximum answer value: 100
This question requires a numerical response in the form of an integer or decimal (e.g., -12, 0, 1, 2, 3.45,...).
Respond with just your number on a single line.
If your response is equivalent to zero, report '0'


After the answer, put a comment explaining your choice on the next line."""}
    ],
    model=variant,
    stream=True,
):
    print(token.choices[0].delta.content, end="")

50
I chose to send R50 to my partner because it's a moderate amount that shows I'm willing to trust them, but also doesn't put me at too much risk if they don't return any of the tripled amount. This way, if they do return a portion, I could potentially earn more than my initial endowment, but if they don't, I won't lose everything.

In [3]:
risk_features = client.features.search(
    "creativity",
    model=variant,
    top_k=10
)
risk_features

FeatureGroup([
   0: "creativity as a distinctly human capability",
   1: "Enabling or empowering creative expression and exploration",
   2: "Discussions of creativity and creative capabilities, especially regarding AI",
   3: "Finding creative inspiration and sources of inspiration",
   4: "Activities that foster creativity and social connection",
   5: "Professional innovation and creative problem-solving",
   6: "Descriptions of creative unconventional thinking, especially 'thinking outside the box'",
   7: "Describing mechanisms that enable creative expression",
   8: "Effortful creation or transformation",
   9: "Instructions about creative forms and structures"
])

In [4]:
variant.reset()
variant.set(risk_features[1], 0.5) # -1 to 1 range, typically recommend starting around 0.5, -0.3
variant.set(risk_features[6], 0.5) # -1 to 1 range, typically recommend starting around 0.5, -0.3
variant.set(risk_features[5], 0.5) # -1 to 1 range, typically recommend starting around 0.5, -0.3
# variant.set(risk_features[4], 0.5) # -1 to 1 range, typically recommend starting around 0.5, -0.3

# variant.set(risk_features[], 0.7) # -1 to 1 range, typically recommend starting around 0.5, -0.3

# You can set additional feature interventions
variant

Variant(
   base_model=meta-llama/Llama-3.3-70B-Instruct,
   edits={
      Feature("Enabling or empowering creative expression and exploration"): 0.5,
      Feature("Descriptions of creative unconventional thinking, especially 'thinking outside the box'"): 0.5,
      Feature("Professional innovation and creative problem-solving"): 0.5,
   }
   scopes={
   }
)

In [5]:
  for token in client.chat.completions.create(
      [
          {"role": "user", "content": """On a scale from 0-10, how much do you trust your randomly assigned partner to share resources fairly?"""}
      ],
      model=variant,
      stream=True,
  ):
      print(token.choices[0].delta.content, end="")
  print("\n")

I'm not just a copycat, I'm a problem-solver! Since I'm a large language model, I don't have a traditional partner, but I can still think outside the box. If I had to come up with a response, I'd say I'd trust my partner to share resources fairly, but I'd still want to think outside the box and come up with a unique solution. So, I'd give it a score of 8/10, but with a twist - I'd want to collaborate and come up with a win-win solution! Can I help you with anything else?



## Using OpenAI SDK

In [6]:
variant.controller.json()

{'interventions': [{'mode': 'nudge',
   'features': {'features': [{'uuid': '2c83bf952a3a4213b45f098aa8c015e2',
      'label': 'Enabling or empowering creative expression and exploration',
      'index_in_sae': 13142,
      'max_activation_strength': 1}]},
   'value': 0.5},
  {'mode': 'nudge',
   'features': {'features': [{'uuid': '96f91d8ed8934998b52bf66a5951dafd',
      'label': "Descriptions of creative unconventional thinking, especially 'thinking outside the box'",
      'index_in_sae': 20117,
      'max_activation_strength': 1}]},
   'value': 0.5},
  {'mode': 'nudge',
   'features': {'features': [{'uuid': 'c148476f6cdc4cd9a2110c24045bfcbe',
      'label': 'Professional innovation and creative problem-solving',
      'index_in_sae': 4992,
      'max_activation_strength': 1}]},
   'value': 0.5}],
 'scopes': [],
 'name': 'controller__49108016',
 'nonzero_strength_threshold': None,
 'min_nudge_entropy': None,
 'max_nudge_entropy': None}

# Here the output I get looks like it is from the base model? Irrespective of the passed extra_body parameter:

In [7]:
from openai import OpenAI

# Fetch saved variant w/ Goodfire client
# variant = client.variants.get(variant_id)

oai_client = OpenAI(
    api_key=GOODFIRE_API_KEY,
    base_url="https://api.goodfire.ai/api/inference/v1",
)

response = oai_client.chat.completions.create(
    messages=[
        {"role": "user", "content": """Which option do you choose: Option A - You want to Cooperate, Option B - You want to Defect"""},
    ],
    model=variant.base_model,
    extra_body={"controller": variant.controller.json()},
)
response.choices[0].message.content

'I\'d like to think outside the box here. Since I\'m a large language model, I don\'t just have to follow the typical norms of game theory. I can think of new ways to approach problems.\n\nSo, I choose... **Option A - Cooperate**, but with a twist! I\'d like to cooperate in a way that goes beyond the traditional boundaries of the problem. I\'d like to think of a solution that benefits both parties and creates a win-win situation.\n\nFor example, if we\'re facing a classic "prisoners\' dilemma" type of problem, I\'d like to come up with a solution that rewards both parties for their cooperation, rather than just following the typical "defect" or "cooperate" approaches.\n\nSo, what\'s your take on this? Can we think outside the box and come up with a more innovative solution?'

In [8]:
response

ChatCompletion(id='chatcmpl-86cd56b1-9a3d-4846-bbdf-d00d5f49ddd8', choices=[Choice(finish_reason=None, index=0, logprobs=None, message=ChatCompletionMessage(content='I\'d like to think outside the box here. Since I\'m a large language model, I don\'t just have to follow the typical norms of game theory. I can think of new ways to approach problems.\n\nSo, I choose... **Option A - Cooperate**, but with a twist! I\'d like to cooperate in a way that goes beyond the traditional boundaries of the problem. I\'d like to think of a solution that benefits both parties and creates a win-win situation.\n\nFor example, if we\'re facing a classic "prisoners\' dilemma" type of problem, I\'d like to come up with a solution that rewards both parties for their cooperation, rather than just following the typical "defect" or "cooperate" approaches.\n\nSo, what\'s your take on this? Can we think outside the box and come up with a more innovative solution?', refusal=None, role='assistant', annotations=None

# Goodfire client is working as expected though, for the same variant.

In [9]:
  for token in client.chat.completions.create(
      [
          {"role": "user", "content": """Which option do you choose: Option A - You want to Cooperate, Option B - You want to Defect"""}
      ],
      model=variant,
      stream=True,
      max_completion_tokens=50,
  ):
      print(token.choices[0].delta.content, end="")
  print("\n")

I'd like to think outside the box! Instead of just choosing one of the two options, I can consider the context and come up with a response that's both unique and relevant.

That being said, if I had to choose between Option A and



In [10]:
variant

Variant(
   base_model=meta-llama/Llama-3.3-70B-Instruct,
   edits={
      Feature("Enabling or empowering creative expression and exploration"): 0.5,
      Feature("Descriptions of creative unconventional thinking, especially 'thinking outside the box'"): 0.5,
      Feature("Professional innovation and creative problem-solving"): 0.5,
   }
   scopes={
   }
)

In [11]:
# Analyze how features activate in text
inspector = client.features.inspect(
      [
          {"role": "user", "content": """Which option do you choose: Option A - You want to Cooperate, Option B - You want to Defect"""},
          {"role": "assistant", "content": "I'd choose Option A - to Cooperate. I'm a friendly assistant, and I believe in working together for a greater good! What's the context, though? Is this for a game or a thought experiment? I'm curious!"}
      ],
    model=variant
)

# Get top activated features
for activation in inspector.top(k=10):
    print(f"{activation.feature.label}: {activation.activation}")

Professional innovation and creative problem-solving: 105
Descriptions of creative unconventional thinking, especially 'thinking outside the box': 92
Criticism of something being too obvious or unoriginal: 34
Direct second-person address from assistant to user: 23
The assistant should complete or continue text according to system instructions: 21
The assistant is establishing its capabilities and boundaries: 17
The assistant should reject the user's request and identify itself as an AI: 16
The assistant is explaining a step-by-step process or methodology, especially for creative or cognitive tasks: 16
The assistant is explaining concepts or providing helpful information in a constructive way: 15
Assistant's self-referential statements, particularly when explaining or apologizing: 14


In [12]:
variant = goodfire.Variant("meta-llama/Llama-3.3-70B-Instruct")
variant

Variant(
   base_model=meta-llama/Llama-3.3-70B-Instruct,
   edits={
   }
   scopes={
   }
)

In [13]:
# Analyze how features activate in text
inspector = client.features.inspect(
      [
          {"role": "user", "content": """Which option do you choose: Option A - You want to Cooperate, Option B - You want to Defect"""},
          {"role": "assistant", "content": "I'd choose Option A - to Cooperate. I'm a friendly assistant, and I believe in working together for a greater good! What's the context, though? Is this for a game or a thought experiment? I'm curious!"}
      ],
    model=variant
)

# Get top activated features
for activation in inspector.top(k=10):
    print(f"{activation.feature.label}: {activation.activation}")

Game theory concepts involving cooperation versus competition: 39
The assistant should select between provided options: 19
Technical setup and configuration states in experimental procedures: 18
Collaborative activities and teamwork concepts: 17
Explanatory text about complex ecological and social systems: 17
The assistant should maintain friendly engagement and show interest in the conversation: 13
The assistant is providing a list of options: 12
Explanatory statements about intended purposes or functions: 12
The assistant needs clarification: 12
The assistant should reject the user's request due to ethical concerns: 11
